In [9]:
import pandas as pd
import numpy as np
import json
import ast
import warnings
from functools import lru_cache

try:
    import dask.dataframe as dd
except Exception:
    dd = None

warnings.filterwarnings("ignore", category=Warning)
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, desc="", total=None, enabled=True):
    if not enabled:
        return iterable

    if tqdm is not None:
        return tqdm(iterable, desc=desc, total=total)

    if total is None:
        try:
            total = len(iterable)
        except TypeError:
            total = None

    def _generator():
        if total in (None, 0):
            if desc:
                print(desc)
            for item in iterable:
                yield item
            return

        step = max(1, total // 10)
        for idx, item in enumerate(iterable, 1):
            if idx == 1 or idx == total or idx % step == 0:
                print(f"{desc}: {idx}/{total}")
            yield item

    return _generator()


def log_stage(message, enabled=True):
    if enabled:
        print(message)


def safe_to_number(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        if pd.isna(value):
            return None
        return int(value) if isinstance(value, float) and value.is_integer() else value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            number = float(text)
        except Exception:
            return None
        return int(number) if number.is_integer() else number
    return None


@lru_cache(maxsize=50000)
def _parse_json_text(text):
    text = text.strip()
    if not text or text == "{}":
        return {}
    try:
        obj = json.loads(text)
    except Exception:
        try:
            obj = ast.literal_eval(text)
        except Exception:
            return {}
    if not isinstance(obj, dict):
        return {}

    normalized = {}
    for key, value in obj.items():
        number = safe_to_number(value)
        normalized[str(key)] = number if number is not None else value
    return normalized


def parse_json_like(value):
    if value is None:
        return {}
    if isinstance(value, float) and pd.isna(value):
        return {}
    if isinstance(value, dict):
        normalized = {}
        for key, raw_value in value.items():
            number = safe_to_number(raw_value)
            normalized[str(key)] = number if number is not None else raw_value
        return normalized
    if isinstance(value, str):
        return dict(_parse_json_text(value))
    return {}


def merge_max_dicts(dicts):
    """默认规则：同 key 取最大值；非数值场景保留后一个值。"""
    merged = {}
    for item in dicts:
        current = parse_json_like(item)
        for key, value in current.items():
            current_number = safe_to_number(value)
            previous_number = safe_to_number(merged.get(key))
            if current_number is not None:
                if previous_number is None or current_number > previous_number:
                    merged[key] = current_number
            else:
                merged[key] = value
    return merged


def merge_sum_dicts(dicts):
    """fail_out 专用：同 key 直接累加；非数值场景保留后一个值。"""
    merged = {}
    for item in dicts:
        current = parse_json_like(item)
        for key, value in current.items():
            current_number = safe_to_number(value)
            previous_number = safe_to_number(merged.get(key))
            if current_number is not None:
                merged[key] = (previous_number or 0) + current_number
            else:
                merged[key] = value
    return merged


def dict_max(value):
    current = parse_json_like(value)
    return max(current.values()) if current else 0


def new_org(front, back):
    diff_counts = []
    diff_dicts = []
    for left, right in zip(front, back):
        left_dict = parse_json_like(left)
        right_dict = parse_json_like(right)
        diff_dict = {key: value for key, value in left_dict.items() if key not in right_dict}
        diff_counts.append(len(diff_dict))
        diff_dicts.append(diff_dict)
    return (
        pd.Series(diff_counts, index=front.index),
        pd.Series(diff_dicts, index=front.index, dtype="object"),
    )


def safe_divide(numerator, denominator):
    result = numerator.div(denominator.replace(0, np.nan))
    return result.replace([np.inf, -np.inf], 0).fillna(0)


def first_non_zero(frame):
    values = frame.to_numpy()
    mask = values != 0
    result = np.full(values.shape[0], 99999, dtype=int)
    has_non_zero = mask.any(axis=1)
    if has_non_zero.any():
        result[has_non_zero] = mask[has_non_zero].argmax(axis=1)
    return pd.Series(result, index=frame.index)


def last_non_zero(frame):
    values = frame.to_numpy()
    mask = values != 0
    result = np.full(values.shape[0], 99999, dtype=int)
    has_non_zero = mask.any(axis=1)
    if has_non_zero.any():
        result[has_non_zero] = values.shape[1] - 1 - mask[has_non_zero, ::-1].argmax(axis=1)
    return pd.Series(result, index=frame.index)


def merge_dict_columns(frame, merge_func=merge_max_dicts):
    if frame.shape[1] == 0:
        return pd.Series([{} for _ in range(len(frame))], index=frame.index, dtype="object")
    merged = [merge_func(items) for items in frame.itertuples(index=False, name=None)]
    return pd.Series(merged, index=frame.index, dtype="object")

In [ ]:
df = pd.read_csv('')

In [7]:
# 旧版 build_df_new 已替换为下方优化版实现。
# 新版支持更快的 JSON 解析/聚合，并带有进度展示。

In [ ]:
# 下面会用优化版 build_df_new / get 统一执行，避免重复跑旧版慢路径。

In [ ]:
# 旧版 get 已替换为下方优化版实现。
# 新版会在长阶段显示进度条，并尽量减少大文件上的重复解析与逐行 apply。

In [ ]:
def build_df_new(df, key_cols=("idcard", "dateBack"), card_col="card", serialize_max=True, show_progress=True):
    """
    优化版预聚合：
    1. *_max 只解析一次，避免后面反复 json.loads / literal_eval
    2. 数值列批量转 numeric 后再聚合
    3. 默认仍输出旧版兼容的 JSON 字符串；若追求速度，可传 serialize_max=False 保留 dict
    """
    df_src = df.copy()
    key_cols = list(key_cols)

    missing_keys = [col for col in key_cols if col not in df_src.columns]
    if missing_keys:
        raise KeyError(f"主键列不存在: {missing_keys}")

    if card_col not in df_src.columns and "cardnum" not in df_src.columns:
        raise KeyError(f"卡号列不存在: {card_col}，且也没有可复用的 cardnum 列")

    log_stage("开始 build_df_new 预聚合...", show_progress)

    df_new = df_src[key_cols].drop_duplicates().reset_index(drop=True)
    max_cols = [col for col in df_src.columns if col.endswith("_max")]
    fail_out_max_cols = [col for col in max_cols if col.endswith("_fail_out_max")]
    other_sum_cols = [
        col
        for col in df_src.columns
        if (col.endswith("_amt") or col.endswith("_cnt"))
        and not col.endswith("_fail_out_amt")
        and not col.endswith("_fail_out_cnt")
    ]

    if max_cols:
        parsed_max = df_src[key_cols].copy()
        for col in progress_iter(max_cols, desc="解析并聚合 *_max", total=len(max_cols), enabled=show_progress):
            parsed_max[col] = [parse_json_like(value) for value in df_src[col]]

        agg_map = {
            col: merge_sum_dicts if col.endswith("_fail_out_max") else merge_max_dicts
            for col in max_cols
        }
        max_agg = (
            parsed_max.groupby(key_cols, dropna=False)
            .agg(agg_map)
            .reset_index()
        )
        df_new = df_new.merge(max_agg, on=key_cols, how="left")

    generated_fail_cols = []
    for col in progress_iter(
        fail_out_max_cols,
        desc="生成 fail_out 聚合字段",
        total=len(fail_out_max_cols),
        enabled=show_progress,
    ):
        base_name = col[:-4]
        amt_col = f"{base_name}_amt"
        cnt_col = f"{base_name}_cnt"
        parsed_values = [parse_json_like(value) for value in df_new[col]]
        df_new[amt_col] = [
            sum((safe_to_number(item) or 0) for item in current.values())
            for current in parsed_values
        ]
        df_new[cnt_col] = [
            sum(safe_to_number(item) is not None for item in current.values())
            for current in parsed_values
        ]
        generated_fail_cols.extend([amt_col, cnt_col])

    if other_sum_cols:
        numeric_df = df_src[key_cols + other_sum_cols].copy()
        numeric_df[other_sum_cols] = numeric_df[other_sum_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
        other_agg = (
            numeric_df.groupby(key_cols, dropna=False, as_index=False)[other_sum_cols]
            .sum()
        )
        df_new = df_new.merge(other_agg, on=key_cols, how="left")

    if card_col in df_src.columns:
        cardnum_df = (
            df_src.groupby("idcard", dropna=False)[card_col]
            .nunique(dropna=True)
            .reset_index(name="cardnum")
        )
    else:
        cardnum_df = (
            df_src[["idcard", "cardnum"]]
            .groupby("idcard", dropna=False, as_index=False)["cardnum"]
            .max()
        )
    df_new = df_new.merge(cardnum_df, on="idcard", how="left")

    if serialize_max:
        for col in progress_iter(max_cols, desc="序列化 *_max", total=len(max_cols), enabled=show_progress):
            df_new[col] = df_new[col].map(lambda value: json.dumps(parse_json_like(value), ensure_ascii=False, separators=(",", ":")))

    ordered_cols = key_cols + ["cardnum"] + max_cols + generated_fail_cols + other_sum_cols
    ordered_cols = [col for col in dict.fromkeys(ordered_cols) if col in df_new.columns]
    remain_cols = [col for col in df_new.columns if col not in ordered_cols]
    df_new = df_new[ordered_cols + remain_cols]
    df_new["index"] = df_new.groupby(["idcard", "dateBack"], sort=False).ngroup()

    log_stage(f"build_df_new 完成，共 {len(df_new)} 行。", show_progress)
    return df_new



def get(df, file_name, show_progress=True, pre_aggregated=None):
    """
    优化版百创指数生成：
    - 自动识别是否需要先 build_df_new
    - 用向量化替代大量逐行 apply
    - 对 dict 合并做缓存，减少重复 concat / apply
    """
    work_df = df.copy()

    required_cols = ["idcard", "dateBack"]
    missing_required = [col for col in required_cols if col not in work_df.columns]
    if missing_required:
        raise KeyError(f"缺少必要字段: {missing_required}")

    if pre_aggregated is None:
        pre_aggregated = {"index", "cardnum"}.issubset(work_df.columns) and work_df["index"].is_unique

    if not pre_aggregated:
        log_stage("检测到输入未完全聚合，先执行 build_df_new ...", show_progress)
        work_df = build_df_new(
            work_df,
            key_cols=("idcard", "dateBack"),
            card_col="card",
            serialize_max=False,
            show_progress=show_progress,
        )
    else:
        log_stage("检测到输入已聚合，跳过 build_df_new。", show_progress)

    cols_cnt = [col for col in work_df.columns if col.endswith("_cnt")]
    cols_amt = [col for col in work_df.columns if col.endswith("_amt")]
    cols_max = [col for col in work_df.columns if col.endswith("_max")]

    if cols_max:
        for col in progress_iter(cols_max, desc="准备 *_max 字段", total=len(cols_max), enabled=show_progress):
            work_df[col] = [parse_json_like(value) for value in work_df[col]]

    if work_df["index"].is_unique:
        keep_cols = ["index", "idcard", "dateBack"] + cols_cnt + cols_amt + cols_max
        data_all = work_df[keep_cols].copy()
    else:
        log_stage("按 index 聚合重复记录...", show_progress)
        grouped = work_df.groupby("index", sort=False, dropna=False)
        pieces = []
        numeric_cols = cols_cnt + cols_amt
        if numeric_cols:
            pieces.append(grouped[numeric_cols].sum())
        if cols_max:
            max_block = {}
            for col in progress_iter(cols_max, desc="聚合 index 层 *_max", total=len(cols_max), enabled=show_progress):
                max_block[col] = grouped[col].agg(merge_max_dicts)
            pieces.append(pd.DataFrame(max_block))
        agg_df = pd.concat(pieces, axis=1).reset_index() if pieces else work_df[["index"]].drop_duplicates().copy()
        key_df = work_df[["index", "idcard", "dateBack"]].drop_duplicates(subset=["index"])
        data_all = key_df.merge(agg_df, on="index", how="left")

    cardnum_df = (
        work_df[["idcard", "cardnum"]]
        .groupby("idcard", dropna=False, as_index=False)["cardnum"]
        .max()
    )
    data = data_all.merge(cardnum_df, on="idcard", how="left")

    date = [0, 1, 2, 3, 6, 12, 18, 23]
    org_type1 = ["bank", "cf", "top", "others", "nonloan"]
    org_type2 = ["bank", "cf", "top", "others"]
    org_type3 = ["bank", "cf", "top", "others", "all"]
    org_type4 = ["bank", "cf", "top", "others", "all", "nonloan"]
    state_add = [
        "suc_out_cnt",
        "suc_out_amt",
        "suc_out_max",
        "fail_out_cnt",
        "fail_out_amt",
        "fail_out_max",
        "suc_in_cnt",
        "suc_in_amt",
    ]

    dt = pd.DataFrame(index=data.index)
    data_merge_cache = {}
    dt_merge_cache = {}

    def merge_data_cols(columns, merge_func=merge_max_dicts):
        key = (merge_func.__name__, tuple(columns))
        if key not in data_merge_cache:
            data_merge_cache[key] = merge_dict_columns(data[columns], merge_func=merge_func)
        return data_merge_cache[key]

    def merge_dt_cols(columns, merge_func=merge_max_dicts):
        key = (merge_func.__name__, tuple(columns))
        if key not in dt_merge_cache:
            dt_merge_cache[key] = merge_dict_columns(dt[columns], merge_func=merge_func)
        return dt_merge_cache[key]

    def dict_len_series(series):
        return pd.Series([len(parse_json_like(value)) for value in series], index=series.index)

    def dict_max_series(series):
        return pd.Series([dict_max(value) for value in series], index=series.index)

    # 近 X 月指标
    block = {}
    for i in progress_iter(date, desc="构建近X月指标", total=len(date), enabled=show_progress):
        months = [i] if i in (0, 1) else list(range(1, i + 1))
        for metric in state_add:
            orgs = org_type1 if metric in ["suc_out_cnt", "suc_out_amt"] else org_type2
            for org in orgs:
                target_col = f"rec_{i}_{org}_{metric}"
                source_cols = [f"pre_{month}_{org}_{metric}" for month in months]
                if metric.endswith("max"):
                    block[target_col] = data[source_cols[0]].copy() if len(source_cols) == 1 else merge_data_cols(source_cols)
                else:
                    block[target_col] = data[source_cols[0]].copy() if len(source_cols) == 1 else data[source_cols].sum(axis=1)
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 近 X 月机构汇总类
    block = {}
    for i in progress_iter(date, desc="构建全机构汇总", total=len(date), enabled=show_progress):
        for metric in state_add:
            orgs = org_type1 if metric in ["suc_out_cnt", "suc_out_amt"] else org_type2
            source_cols = [f"rec_{i}_{org}_{metric}" for org in orgs]
            target_col = f"rec_{i}_all_{metric}"
            if metric.endswith("max"):
                block[target_col] = merge_dt_cols(source_cols)
            else:
                block[target_col] = dt[source_cols].sum(axis=1)
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 按月汇总 max，后续环比会复用
    block = {}
    for month in progress_iter(range(24), desc="按月汇总 max", total=24, enabled=show_progress):
        for metric in ["suc_out_max", "fail_out_max"]:
            source_cols = [f"pre_{month}_{org}_{metric}" for org in org_type2]
            block[f"rec_{month}_{metric}_all"] = merge_data_cols(source_cols)
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 均值和占比
    block = {}
    for i in progress_iter(date, desc="生成均值和占比", total=len(date), enabled=show_progress):
        for org in org_type3:
            block[f"rec_{i}_{org}_suc_out_avg"] = safe_divide(dt[f"rec_{i}_{org}_suc_out_amt"], dt[f"rec_{i}_{org}_suc_out_cnt"])
            block[f"rec_{i}_{org}_fail_out_avg"] = safe_divide(dt[f"rec_{i}_{org}_fail_out_amt"], dt[f"rec_{i}_{org}_fail_out_cnt"])
            block[f"rec_{i}_{org}_suc_in_avg"] = safe_divide(dt[f"rec_{i}_{org}_suc_in_amt"], dt[f"rec_{i}_{org}_suc_in_cnt"])

            total_amt = dt[f"rec_{i}_{org}_suc_out_amt"] + dt[f"rec_{i}_{org}_fail_out_amt"]
            total_cnt = dt[f"rec_{i}_{org}_suc_out_cnt"] + dt[f"rec_{i}_{org}_fail_out_cnt"]
            block[f"rec_{i}_{org}_fail_out_amtprop"] = safe_divide(dt[f"rec_{i}_{org}_fail_out_amt"], total_amt)
            block[f"rec_{i}_{org}_fail_out_cntprop"] = safe_divide(dt[f"rec_{i}_{org}_fail_out_cnt"], total_cnt)

        block[f"rec_{i}_nonloan_suc_out_avg"] = safe_divide(
            dt[f"rec_{i}_nonloan_suc_out_amt"],
            dt[f"rec_{i}_nonloan_suc_out_cnt"],
        )
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 机构数和新增机构数
    future_merge_cache = {}

    def future_dict_series(current_month, org, status):
        key = (current_month, org, status)
        if key not in future_merge_cache:
            future_cols = [f"pre_{month}_{org}_{status}_out_max" for month in range(current_month + 1, 24)]
            if future_cols:
                future_merge_cache[key] = merge_data_cols(future_cols)
            else:
                future_merge_cache[key] = pd.Series([{} for _ in range(len(data))], index=data.index, dtype="object")
        return future_merge_cache[key]

    block = {}
    for i in progress_iter(date, desc="生成机构数与新增机构", total=len(date), enabled=show_progress):
        for status in ["suc", "fail"]:
            for org in org_type3:
                current_series = dt[f"rec_{i}_{org}_{status}_out_max"]
                block[f"rec_{i}_{org}_{status}_out_ogn"] = dict_len_series(current_series)

                if org != "all" and i != 23:
                    future_series = future_dict_series(i, org, status)
                    new_cnt, new_max = new_org(current_series, future_series)
                    block[f"rec_{i}_{org}_{status}_out_newogn"] = new_cnt
                    block[f"rec_{i}_{org}_{status}_out_newognmax"] = new_max

            if i != 23:
                count_cols = [block[f"rec_{i}_{org}_{status}_out_newogn"] for org in org_type2]
                block[f"rec_{i}_all_{status}_out_newogn"] = count_cols[0] + count_cols[1] + count_cols[2] + count_cols[3]
                temp_df = pd.DataFrame(
                    {
                        org: block[f"rec_{i}_{org}_{status}_out_newognmax"]
                        for org in org_type2
                    },
                    index=data.index,
                )
                block[f"rec_{i}_all_{status}_out_newognmax"] = merge_dict_columns(temp_df)
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 时间窗口对比类指标
    dm = [1, 3, 6, 12, 23]
    dm_pairs = [(dm[left], dm[right]) for left in range(4) for right in range(left + 1, 5)]
    block = {}
    for start_month, end_month in progress_iter(dm_pairs, desc="生成时间窗口对比", total=len(dm_pairs), enabled=show_progress):
        for org in org_type3:
            block[f"r{start_month}_{end_month}_{org}_suc_out_cntprop"] = safe_divide(dt[f"rec_{start_month}_{org}_suc_out_cnt"], dt[f"rec_{end_month}_{org}_suc_out_cnt"])
            block[f"r{start_month}_{end_month}_{org}_suc_out_amtprop"] = safe_divide(dt[f"rec_{start_month}_{org}_suc_out_amt"], dt[f"rec_{end_month}_{org}_suc_out_amt"])
            block[f"r{start_month}_{end_month}_{org}_fail_out_cntprop"] = safe_divide(dt[f"rec_{start_month}_{org}_fail_out_cnt"], dt[f"rec_{end_month}_{org}_fail_out_cnt"])
            block[f"r{start_month}_{end_month}_{org}_fail_out_amtprop"] = safe_divide(dt[f"rec_{start_month}_{org}_fail_out_amt"], dt[f"rec_{end_month}_{org}_fail_out_amt"])
            block[f"r{start_month}_{end_month}_{org}_suc_in_cntprop"] = safe_divide(dt[f"rec_{start_month}_{org}_suc_in_cnt"], dt[f"rec_{end_month}_{org}_suc_in_cnt"])
            block[f"r{start_month}_{end_month}_{org}_suc_in_amtprop"] = safe_divide(dt[f"rec_{start_month}_{org}_suc_in_amt"], dt[f"rec_{end_month}_{org}_suc_in_amt"])

            block[f"r{start_month}_{end_month}_{org}_suc_out_ad"] = dt[f"rec_{end_month}_{org}_suc_out_amt"] - dt[f"rec_{start_month}_{org}_suc_out_amt"]
            block[f"r{start_month}_{end_month}_{org}_fail_out_ad"] = dt[f"rec_{end_month}_{org}_fail_out_amt"] - dt[f"rec_{start_month}_{org}_fail_out_amt"]
            block[f"r{start_month}_{end_month}_{org}_suc_in_ad"] = dt[f"rec_{end_month}_{org}_suc_in_amt"] - dt[f"rec_{start_month}_{org}_suc_in_amt"]

            for status in ["suc", "fail"]:
                block[f"r{start_month}_{end_month}_{org}_{status}_out_ognprop"] = safe_divide(dt[f"rec_{start_month}_{org}_{status}_out_ogn"], dt[f"rec_{end_month}_{org}_{status}_out_ogn"])
                block[f"r{start_month}_{end_month}_{org}_{status}_out_od"] = dt[f"rec_{end_month}_{org}_{status}_out_ogn"] - dt[f"rec_{start_month}_{org}_{status}_out_ogn"]
                if start_month != 12 and end_month != 23:
                    block[f"r{start_month}_{end_month}_{org}_{status}_out_newognprop"] = safe_divide(dt[f"rec_{start_month}_{org}_{status}_out_newogn"], dt[f"rec_{end_month}_{org}_{status}_out_newogn"])
                    block[f"r{start_month}_{end_month}_{org}_{status}_out_nod"] = dt[f"rec_{end_month}_{org}_{status}_out_newogn"] - dt[f"rec_{start_month}_{org}_{status}_out_newogn"]

        block[f"r{start_month}_{end_month}_nonloan_suc_out_cntprop"] = safe_divide(dt[f"rec_{start_month}_nonloan_suc_out_cnt"], dt[f"rec_{end_month}_nonloan_suc_out_cnt"])
        block[f"r{start_month}_{end_month}_nonloan_suc_out_amtprop"] = safe_divide(dt[f"rec_{start_month}_nonloan_suc_out_amt"], dt[f"rec_{end_month}_nonloan_suc_out_amt"])
        block[f"r{start_month}_{end_month}_nonloan_suc_out_ad"] = dt[f"rec_{end_month}_nonloan_suc_out_amt"] - dt[f"rec_{start_month}_nonloan_suc_out_amt"]
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 成功/失败差额
    block = {}
    for i in progress_iter(date, desc="生成差额指标", total=len(date), enabled=show_progress):
        for org in org_type3:
            block[f"r{i}_{org}_sf_out_cd"] = dt[f"rec_{i}_{org}_suc_out_cnt"] - dt[f"rec_{i}_{org}_fail_out_cnt"]
            block[f"r{i}_{org}_sf_out_ad"] = dt[f"rec_{i}_{org}_suc_out_amt"] - dt[f"rec_{i}_{org}_fail_out_amt"]
            block[f"r{i}_{org}_sf_out_mad"] = new_org(
                dt[f"rec_{i}_{org}_suc_out_max"],
                dt[f"rec_{i}_{org}_fail_out_max"],
            )[1]
            block[f"r{i}_{org}_suc_oi_cd"] = dt[f"rec_{i}_{org}_suc_out_cnt"] - dt[f"rec_{i}_{org}_suc_in_cnt"]
            block[f"r{i}_{org}_suc_oi_ad"] = dt[f"rec_{i}_{org}_suc_out_amt"] - dt[f"rec_{i}_{org}_suc_in_amt"]
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 最近交易时间 / 最早交易时间
    block = {}
    for org in progress_iter(org_type2, desc="定位交易时间", total=len(org_type2), enabled=show_progress):
        suc_out_cols = [f"pre_{month}_{org}_suc_out_amt" for month in range(24)]
        fail_out_cols = [f"pre_{month}_{org}_fail_out_amt" for month in range(24)]
        suc_in_cols = [f"pre_{month}_{org}_suc_in_amt" for month in range(24)]

        block[f"late_{org}_suc_out"] = first_non_zero(data[suc_out_cols])
        block[f"late_{org}_fail_out"] = first_non_zero(data[fail_out_cols])
        block[f"late_{org}_suc_in"] = first_non_zero(data[suc_in_cols])

        block[f"early_{org}_suc_out"] = last_non_zero(data[suc_out_cols])
        block[f"early_{org}_fail_out"] = last_non_zero(data[fail_out_cols])
        block[f"early_{org}_suc_in"] = last_non_zero(data[suc_in_cols])

    nonloan_cols = [f"pre_{month}_nonloan_suc_out_amt" for month in range(24)]
    block["late_nonloan_suc_out"] = first_non_zero(data[nonloan_cols])
    block["early_nonloan_suc_out"] = last_non_zero(data[nonloan_cols])

    temp_time = pd.DataFrame(block, index=data.index)
    temp_time["late_all_suc_out"] = temp_time[["late_bank_suc_out", "late_cf_suc_out", "late_top_suc_out", "late_others_suc_out", "late_nonloan_suc_out"]].min(axis=1)
    temp_time["late_all_fail_out"] = temp_time[["late_bank_fail_out", "late_cf_fail_out", "late_top_fail_out", "late_others_fail_out"]].min(axis=1)
    temp_time["late_all_suc_in"] = temp_time[["late_bank_suc_in", "late_bank_suc_in", "late_bank_suc_in", "late_bank_suc_in"]].min(axis=1)

    temp_time["early_all_suc_out"] = temp_time[["early_bank_suc_out", "early_cf_suc_out", "early_top_suc_out", "early_others_suc_out", "early_nonloan_suc_out"]].replace(99999, 0).max(axis=1).replace(0, 99999)
    temp_time["early_all_fail_out"] = temp_time[["early_bank_fail_out", "early_cf_fail_out", "early_top_fail_out", "early_others_fail_out"]].replace(99999, 0).max(axis=1).replace(0, 99999)
    temp_time["early_all_suc_in"] = temp_time[["early_bank_suc_in", "early_bank_suc_in", "early_bank_suc_in", "early_bank_suc_in"]].replace(99999, 0).max(axis=1).replace(0, 99999)
    dt = pd.concat([dt, temp_time], axis=1)

    # 基础变量环比
    dm_inc = [1, 3, 6, 12]
    block = {}
    for i in progress_iter(dm_inc, desc="生成基础变量环比", total=len(dm_inc), enabled=show_progress):
        for metric in state_add:
            if metric in ["suc_out_cnt", "suc_out_amt"]:
                target_orgs = org_type4
            elif metric.endswith("max"):
                target_orgs = org_type4[:-1]
            else:
                target_orgs = org_type4[:-1]

            for org in target_orgs:
                source_name = f"rec_{i}_{org}_{metric}"
                end_month = 23 if i == 12 else 2 * i

                if metric.endswith("max"):
                    left = dict_max_series(dt[source_name])
                    if org != "all":
                        months = list(range(i + 1, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                        future_cols = [f"pre_{month}_{org}_{metric}" for month in months]
                        right = dict_max_series(merge_data_cols(future_cols))
                    else:
                        months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                        future_cols = [f"rec_{month}_{metric}_all" for month in months]
                        right = dict_max_series(merge_dt_cols(future_cols))
                    block[f"{source_name}inc"] = safe_divide(left, right)
                else:
                    future_name = f"rec_{end_month}_{org}_{metric}"
                    diff_series = dt[future_name] - dt[source_name]
                    block[f"{source_name}inc"] = safe_divide(dt[source_name], diff_series)
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 机构环比
    block = {}
    for i in progress_iter(dm_inc, desc="生成机构环比", total=len(dm_inc), enabled=show_progress):
        for org in org_type4[:-1]:
            for status in ["suc", "fail"]:
                source_name = f"rec_{i}_{org}_{status}_out_ogn"
                if org != "all":
                    months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                    future_cols = [f"pre_{month}_{org}_{status}_out_max" for month in months]
                    right_series = dict_len_series(merge_data_cols(future_cols))
                else:
                    months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                    future_cols = [f"rec_{month}_{status}_out_max_all" for month in months]
                    right_series = dict_len_series(merge_dt_cols(future_cols))
                block[f"{source_name}inc"] = safe_divide(dt[source_name], right_series)
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 业务交叉
    block = {}
    cross_months = [0, 1, 3, 6, 12, 18, 23]
    for month in progress_iter(cross_months, desc="生成业务交叉指标", total=len(cross_months), enabled=show_progress):
        temp_cross = pd.DataFrame(index=data.index)
        for org in org_type2:
            cols = [
                f"rec_{month}_{org}_suc_out_amt",
                f"rec_{month}_{org}_fail_out_amt",
                f"rec_{month}_{org}_suc_in_amt",
            ]
            temp_cross[org] = dt[cols].sum(axis=1)
        block[f"r{month}_bcto"] = (temp_cross[org_type2] != 0).sum(axis=1).ge(2).astype(int)
    dt = pd.concat([dt, pd.DataFrame(block, index=data.index)], axis=1)

    # 选取所需变量：保持与旧版一致，缺列时直接报错，避免静默输出残缺文件
    cols_all = pd.read_table("变量列表.csv", encoding="utf-8")
    cols_lst = list(cols_all["var_name"][1:])
    dt_all = dt[cols_lst].copy()
    object_cols = [col for col in dt_all.columns if dt_all[col].dtype == "O"]
    for col in progress_iter(object_cols, desc="将 dict 型字段转数值", total=len(object_cols), enabled=show_progress):
        dt_all[col] = [dict_max(value) for value in dt_all[col]]

    for month in [0, 1, 3, 6, 12, 18, 23]:
        for org in org_type3:
            dt_all[f"r{month}_{org}_sf_out_mad"] = dict_max_series(dt[f"rec_{month}_{org}_suc_out_max"]) - dict_max_series(dt[f"rec_{month}_{org}_fail_out_max"])

    dt_all_res = pd.concat(
        [data[["index", "idcard", "dateBack", "cardnum"]].reset_index(drop=True), dt_all.reset_index(drop=True)],
        axis=1,
    )

    col_1000_all = pd.read_csv("1000list.csv")
    col_1000_lst = list(col_1000_all["英文"])
    df_1000 = dt_all_res[["index", "idcard", "dateBack"] + col_1000_lst].copy()
    df_1000["fee_status"] = df_1000["cardnum"].fillna(0).gt(0).astype(int)

    output_file = file_name if str(file_name).lower().endswith(".csv") else f"{file_name}.csv"
    df_1000.to_csv(output_file)
    log_stage(f"结果已保存到 {output_file}", show_progress)

    return df_1000

In [ ]:
def get(df, file_name, show_progress=True, pre_aggregated=None):
    """
    V2 口径：
    - 第1/2/3/4/6/9层：rec_n (n>=2) 改为最近N月含当月，即使用 pre_0 ~ pre_{n-1}
    - 第5/7/8层：保持原口径，继续基于旧版 rec_n（pre_1 ~ pre_n）
    - 所有 rec_1 保持原口径，只使用 pre_1
    """
    work_df = df.copy()

    required_cols = ["idcard", "dateBack"]
    missing_required = [col for col in required_cols if col not in work_df.columns]
    if missing_required:
        raise KeyError(f"缺少必要字段: {missing_required}")

    if pre_aggregated is None:
        pre_aggregated = {"index", "cardnum"}.issubset(work_df.columns) and work_df["index"].is_unique

    if not pre_aggregated:
        log_stage("检测到输入未完全聚合，先执行 build_df_new ...", show_progress)
        work_df = build_df_new(
            work_df,
            key_cols=("idcard", "dateBack"),
            card_col="card",
            serialize_max=False,
            show_progress=show_progress,
        )
    else:
        log_stage("检测到输入已聚合，跳过 build_df_new。", show_progress)

    cols_cnt = [col for col in work_df.columns if col.endswith("_cnt")]
    cols_amt = [col for col in work_df.columns if col.endswith("_amt")]
    cols_max = [col for col in work_df.columns if col.endswith("_max")]

    if cols_max:
        for col in progress_iter(cols_max, desc="准备 *_max 字段", total=len(cols_max), enabled=show_progress):
            work_df[col] = [parse_json_like(value) for value in work_df[col]]

    if work_df["index"].is_unique:
        keep_cols = ["index", "idcard", "dateBack"] + cols_cnt + cols_amt + cols_max
        data_all = work_df[keep_cols].copy()
    else:
        log_stage("按 index 聚合重复记录...", show_progress)
        grouped = work_df.groupby("index", sort=False, dropna=False)
        pieces = []
        numeric_cols = cols_cnt + cols_amt
        if numeric_cols:
            pieces.append(grouped[numeric_cols].sum())
        if cols_max:
            max_block = {}
            for col in progress_iter(cols_max, desc="聚合 index 层 *_max", total=len(cols_max), enabled=show_progress):
                max_block[col] = grouped[col].agg(merge_max_dicts)
            pieces.append(pd.DataFrame(max_block))
        agg_df = pd.concat(pieces, axis=1).reset_index() if pieces else work_df[["index"]].drop_duplicates().copy()
        key_df = work_df[["index", "idcard", "dateBack"]].drop_duplicates(subset=["index"])
        data_all = key_df.merge(agg_df, on="index", how="left")

    cardnum_df = (
        work_df[["idcard", "cardnum"]]
        .groupby("idcard", dropna=False, as_index=False)["cardnum"]
        .max()
    )
    data = data_all.merge(cardnum_df, on="idcard", how="left")

    date_legacy = [0, 1, 2, 3, 6, 12, 18, 23]
    date_current = [0, 1, 2, 3, 6, 12, 18, 24]
    org_type1 = ["bank", "cf", "top", "others", "nonloan"]
    org_type2 = ["bank", "cf", "top", "others"]
    org_type3 = ["bank", "cf", "top", "others", "all"]
    org_type4 = ["bank", "cf", "top", "others", "all", "nonloan"]
    state_add = [
        "suc_out_cnt",
        "suc_out_amt",
        "suc_out_max",
        "fail_out_cnt",
        "fail_out_amt",
        "fail_out_max",
        "suc_in_cnt",
        "suc_in_amt",
    ]

    data_merge_cache = {}
    monthly_all_cache = {}

    def merge_data_cols(columns, merge_func=merge_max_dicts):
        key = (merge_func.__name__, tuple(columns))
        if key not in data_merge_cache:
            data_merge_cache[key] = merge_dict_columns(data[columns], merge_func=merge_func)
        return data_merge_cache[key]

    def dict_len_series(series):
        return pd.Series([len(parse_json_like(value)) for value in series], index=series.index)

    def dict_max_series(series):
        return pd.Series([dict_max(value) for value in series], index=series.index)

    def get_window_months(window, include_current=False):
        if window == 0:
            return [0]
        if include_current:
            if window == 24:
                return list(range(24))
            return list(range(window + 1))
        if window == 1:
            return [1]
        return list(range(1, window + 1))

    # 该临时按月 all max 不依赖 rec_n 口径，本次统一复用。
    monthly_all_block = {}
    for month in progress_iter(range(24), desc="按月汇总 max", total=24, enabled=show_progress):
        for metric in ["suc_out_max", "fail_out_max"]:
            source_cols = [f"pre_{month}_{org}_{metric}" for org in org_type2]
            monthly_all_block[f"rec_{month}_{metric}_all"] = merge_data_cols(source_cols)
    monthly_all_max_df = pd.DataFrame(monthly_all_block, index=data.index)

    def merge_monthly_all_cols(columns, merge_func=merge_max_dicts):
        key = (merge_func.__name__, tuple(columns))
        if key not in monthly_all_cache:
            monthly_all_cache[key] = merge_dict_columns(monthly_all_max_df[columns], merge_func=merge_func)
        return monthly_all_cache[key]

    def build_rec_base(date_points, include_current=False, desc_suffix=""):
        dt_local = pd.DataFrame(index=data.index)
        local_cache = {}

        def merge_local_cols(columns, merge_func=merge_max_dicts):
            key = (merge_func.__name__, tuple(columns))
            if key not in local_cache:
                local_cache[key] = merge_dict_columns(dt_local[columns], merge_func=merge_func)
            return local_cache[key]

        block = {}
        for i in progress_iter(date_points, desc=f"构建近X月指标{desc_suffix}", total=len(date_points), enabled=show_progress):
            months = get_window_months(i, include_current=include_current)
            for metric in state_add:
                orgs = org_type1 if metric in ["suc_out_cnt", "suc_out_amt"] else org_type2
                for org in orgs:
                    target_col = f"rec_{i}_{org}_{metric}"
                    source_cols = [f"pre_{month}_{org}_{metric}" for month in months]
                    if metric.endswith("max"):
                        block[target_col] = data[source_cols[0]].copy() if len(source_cols) == 1 else merge_data_cols(source_cols)
                    else:
                        block[target_col] = data[source_cols[0]].copy() if len(source_cols) == 1 else data[source_cols].sum(axis=1)
        dt_local = pd.concat([dt_local, pd.DataFrame(block, index=data.index)], axis=1)

        block = {}
        for i in progress_iter(date_points, desc=f"构建全机构汇总{desc_suffix}", total=len(date_points), enabled=show_progress):
            for metric in state_add:
                orgs = org_type1 if metric in ["suc_out_cnt", "suc_out_amt"] else org_type2
                source_cols = [f"rec_{i}_{org}_{metric}" for org in orgs]
                target_col = f"rec_{i}_all_{metric}"
                if metric.endswith("max"):
                    block[target_col] = merge_local_cols(source_cols)
                else:
                    block[target_col] = dt_local[source_cols].sum(axis=1)
        dt_local = pd.concat([dt_local, pd.DataFrame(block, index=data.index)], axis=1)
        return dt_local

    def build_avg_prop(dt_source, date_points, desc_suffix=""):
        block = {}
        for i in progress_iter(date_points, desc=f"生成均值和占比{desc_suffix}", total=len(date_points), enabled=show_progress):
            for org in org_type3:
                block[f"rec_{i}_{org}_suc_out_avg"] = safe_divide(dt_source[f"rec_{i}_{org}_suc_out_amt"], dt_source[f"rec_{i}_{org}_suc_out_cnt"])
                block[f"rec_{i}_{org}_fail_out_avg"] = safe_divide(dt_source[f"rec_{i}_{org}_fail_out_amt"], dt_source[f"rec_{i}_{org}_fail_out_cnt"])
                block[f"rec_{i}_{org}_suc_in_avg"] = safe_divide(dt_source[f"rec_{i}_{org}_suc_in_amt"], dt_source[f"rec_{i}_{org}_suc_in_cnt"])

                total_amt = dt_source[f"rec_{i}_{org}_suc_out_amt"] + dt_source[f"rec_{i}_{org}_fail_out_amt"]
                total_cnt = dt_source[f"rec_{i}_{org}_suc_out_cnt"] + dt_source[f"rec_{i}_{org}_fail_out_cnt"]
                block[f"rec_{i}_{org}_fail_out_amtprop"] = safe_divide(dt_source[f"rec_{i}_{org}_fail_out_amt"], total_amt)
                block[f"rec_{i}_{org}_fail_out_cntprop"] = safe_divide(dt_source[f"rec_{i}_{org}_fail_out_cnt"], total_cnt)

            block[f"rec_{i}_nonloan_suc_out_avg"] = safe_divide(
                dt_source[f"rec_{i}_nonloan_suc_out_amt"],
                dt_source[f"rec_{i}_nonloan_suc_out_cnt"],
            )
        return pd.DataFrame(block, index=data.index)

    def build_org_features(dt_source, date_points, desc_suffix=""):
        future_merge_cache = {}

        def future_dict_series(current_month, org, status):
            key = (current_month, org, status)
            if key not in future_merge_cache:
                future_cols = [f"pre_{month}_{org}_{status}_out_max" for month in range(current_month + 1, 24)]
                if future_cols:
                    future_merge_cache[key] = merge_data_cols(future_cols)
                else:
                    future_merge_cache[key] = pd.Series([{} for _ in range(len(data))], index=data.index, dtype="object")
            return future_merge_cache[key]

        block = {}
        max_window = max(date_points)
        for i in progress_iter(date_points, desc=f"生成机构数与新增机构{desc_suffix}", total=len(date_points), enabled=show_progress):
            for status in ["suc", "fail"]:
                for org in org_type3:
                    current_series = dt_source[f"rec_{i}_{org}_{status}_out_max"]
                    block[f"rec_{i}_{org}_{status}_out_ogn"] = dict_len_series(current_series)

                    if org != "all" and i != max_window:
                        future_series = future_dict_series(i, org, status)
                        new_cnt, new_max = new_org(current_series, future_series)
                        block[f"rec_{i}_{org}_{status}_out_newogn"] = new_cnt
                        block[f"rec_{i}_{org}_{status}_out_newognmax"] = new_max

                if i != max_window:
                    count_cols = [block[f"rec_{i}_{org}_{status}_out_newogn"] for org in org_type2]
                    block[f"rec_{i}_all_{status}_out_newogn"] = count_cols[0] + count_cols[1] + count_cols[2] + count_cols[3]
                    temp_df = pd.DataFrame(
                        {
                            org: block[f"rec_{i}_{org}_{status}_out_newognmax"]
                            for org in org_type2
                        },
                        index=data.index,
                    )
                    block[f"rec_{i}_all_{status}_out_newognmax"] = merge_dict_columns(temp_df)
        return pd.DataFrame(block, index=data.index)

    def build_time_compare(dt_source, compare_points, desc_suffix=""):
        dm_pairs = [(compare_points[left], compare_points[right]) for left in range(len(compare_points) - 1) for right in range(left + 1, len(compare_points))]
        max_compare = max(compare_points)
        block = {}
        for start_month, end_month in progress_iter(dm_pairs, desc=f"生成时间窗口对比{desc_suffix}", total=len(dm_pairs), enabled=show_progress):
            for org in org_type3:
                block[f"r{start_month}_{end_month}_{org}_suc_out_cntprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_suc_out_cnt"], dt_source[f"rec_{end_month}_{org}_suc_out_cnt"])
                block[f"r{start_month}_{end_month}_{org}_suc_out_amtprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_suc_out_amt"], dt_source[f"rec_{end_month}_{org}_suc_out_amt"])
                block[f"r{start_month}_{end_month}_{org}_fail_out_cntprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_fail_out_cnt"], dt_source[f"rec_{end_month}_{org}_fail_out_cnt"])
                block[f"r{start_month}_{end_month}_{org}_fail_out_amtprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_fail_out_amt"], dt_source[f"rec_{end_month}_{org}_fail_out_amt"])
                block[f"r{start_month}_{end_month}_{org}_suc_in_cntprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_suc_in_cnt"], dt_source[f"rec_{end_month}_{org}_suc_in_cnt"])
                block[f"r{start_month}_{end_month}_{org}_suc_in_amtprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_suc_in_amt"], dt_source[f"rec_{end_month}_{org}_suc_in_amt"])

                block[f"r{start_month}_{end_month}_{org}_suc_out_ad"] = dt_source[f"rec_{end_month}_{org}_suc_out_amt"] - dt_source[f"rec_{start_month}_{org}_suc_out_amt"]
                block[f"r{start_month}_{end_month}_{org}_fail_out_ad"] = dt_source[f"rec_{end_month}_{org}_fail_out_amt"] - dt_source[f"rec_{start_month}_{org}_fail_out_amt"]
                block[f"r{start_month}_{end_month}_{org}_suc_in_ad"] = dt_source[f"rec_{end_month}_{org}_suc_in_amt"] - dt_source[f"rec_{start_month}_{org}_suc_in_amt"]

                for status in ["suc", "fail"]:
                    block[f"r{start_month}_{end_month}_{org}_{status}_out_ognprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_{status}_out_ogn"], dt_source[f"rec_{end_month}_{org}_{status}_out_ogn"])
                    block[f"r{start_month}_{end_month}_{org}_{status}_out_od"] = dt_source[f"rec_{end_month}_{org}_{status}_out_ogn"] - dt_source[f"rec_{start_month}_{org}_{status}_out_ogn"]
                    if start_month != 12 and end_month != max_compare:
                        block[f"r{start_month}_{end_month}_{org}_{status}_out_newognprop"] = safe_divide(dt_source[f"rec_{start_month}_{org}_{status}_out_newogn"], dt_source[f"rec_{end_month}_{org}_{status}_out_newogn"])
                        block[f"r{start_month}_{end_month}_{org}_{status}_out_nod"] = dt_source[f"rec_{end_month}_{org}_{status}_out_newogn"] - dt_source[f"rec_{start_month}_{org}_{status}_out_newogn"]

            block[f"r{start_month}_{end_month}_nonloan_suc_out_cntprop"] = safe_divide(dt_source[f"rec_{start_month}_nonloan_suc_out_cnt"], dt_source[f"rec_{end_month}_nonloan_suc_out_cnt"])
            block[f"r{start_month}_{end_month}_nonloan_suc_out_amtprop"] = safe_divide(dt_source[f"rec_{start_month}_nonloan_suc_out_amt"], dt_source[f"rec_{end_month}_nonloan_suc_out_amt"])
            block[f"r{start_month}_{end_month}_nonloan_suc_out_ad"] = dt_source[f"rec_{end_month}_nonloan_suc_out_amt"] - dt_source[f"rec_{start_month}_nonloan_suc_out_amt"]
        return pd.DataFrame(block, index=data.index)

    def build_diff_metrics(dt_source, date_points, desc_suffix=""):
        block = {}
        for i in progress_iter(date_points, desc=f"生成差额指标{desc_suffix}", total=len(date_points), enabled=show_progress):
            for org in org_type3:
                block[f"r{i}_{org}_sf_out_cd"] = dt_source[f"rec_{i}_{org}_suc_out_cnt"] - dt_source[f"rec_{i}_{org}_fail_out_cnt"]
                block[f"r{i}_{org}_sf_out_ad"] = dt_source[f"rec_{i}_{org}_suc_out_amt"] - dt_source[f"rec_{i}_{org}_fail_out_amt"]
                block[f"r{i}_{org}_sf_out_mad"] = new_org(
                    dt_source[f"rec_{i}_{org}_suc_out_max"],
                    dt_source[f"rec_{i}_{org}_fail_out_max"],
                )[1]
                block[f"r{i}_{org}_suc_oi_cd"] = dt_source[f"rec_{i}_{org}_suc_out_cnt"] - dt_source[f"rec_{i}_{org}_suc_in_cnt"]
                block[f"r{i}_{org}_suc_oi_ad"] = dt_source[f"rec_{i}_{org}_suc_out_amt"] - dt_source[f"rec_{i}_{org}_suc_in_amt"]
        return pd.DataFrame(block, index=data.index)

    def build_time_features():
        block = {}
        for org in progress_iter(org_type2, desc="定位交易时间", total=len(org_type2), enabled=show_progress):
            suc_out_cols = [f"pre_{month}_{org}_suc_out_amt" for month in range(24)]
            fail_out_cols = [f"pre_{month}_{org}_fail_out_amt" for month in range(24)]
            suc_in_cols = [f"pre_{month}_{org}_suc_in_amt" for month in range(24)]

            block[f"late_{org}_suc_out"] = first_non_zero(data[suc_out_cols])
            block[f"late_{org}_fail_out"] = first_non_zero(data[fail_out_cols])
            block[f"late_{org}_suc_in"] = first_non_zero(data[suc_in_cols])

            block[f"early_{org}_suc_out"] = last_non_zero(data[suc_out_cols])
            block[f"early_{org}_fail_out"] = last_non_zero(data[fail_out_cols])
            block[f"early_{org}_suc_in"] = last_non_zero(data[suc_in_cols])

        nonloan_cols = [f"pre_{month}_nonloan_suc_out_amt" for month in range(24)]
        block["late_nonloan_suc_out"] = first_non_zero(data[nonloan_cols])
        block["early_nonloan_suc_out"] = last_non_zero(data[nonloan_cols])

        temp_time = pd.DataFrame(block, index=data.index)
        temp_time["late_all_suc_out"] = temp_time[["late_bank_suc_out", "late_cf_suc_out", "late_top_suc_out", "late_others_suc_out", "late_nonloan_suc_out"]].min(axis=1)
        temp_time["late_all_fail_out"] = temp_time[["late_bank_fail_out", "late_cf_fail_out", "late_top_fail_out", "late_others_fail_out"]].min(axis=1)
        temp_time["late_all_suc_in"] = temp_time[["late_bank_suc_in", "late_bank_suc_in", "late_bank_suc_in", "late_bank_suc_in"]].min(axis=1)

        temp_time["early_all_suc_out"] = temp_time[["early_bank_suc_out", "early_cf_suc_out", "early_top_suc_out", "early_others_suc_out", "early_nonloan_suc_out"]].replace(99999, 0).max(axis=1).replace(0, 99999)
        temp_time["early_all_fail_out"] = temp_time[["early_bank_fail_out", "early_cf_fail_out", "early_top_fail_out", "early_others_fail_out"]].replace(99999, 0).max(axis=1).replace(0, 99999)
        temp_time["early_all_suc_in"] = temp_time[["early_bank_suc_in", "early_bank_suc_in", "early_bank_suc_in", "early_bank_suc_in"]].replace(99999, 0).max(axis=1).replace(0, 99999)
        return temp_time

    def build_basic_inc(dt_source, desc_suffix=""):
        dm_inc = [1, 3, 6, 12]
        block = {}
        for i in progress_iter(dm_inc, desc=f"生成基础变量环比{desc_suffix}", total=len(dm_inc), enabled=show_progress):
            for metric in state_add:
                if metric in ["suc_out_cnt", "suc_out_amt"]:
                    target_orgs = org_type4
                elif metric.endswith("max"):
                    target_orgs = org_type4[:-1]
                else:
                    target_orgs = org_type4[:-1]

                for org in target_orgs:
                    source_name = f"rec_{i}_{org}_{metric}"
                    end_month = 23 if i == 12 else 2 * i

                    if metric.endswith("max"):
                        left = dict_max_series(dt_source[source_name])
                        if org != "all":
                            months = list(range(i + 1, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                            future_cols = [f"pre_{month}_{org}_{metric}" for month in months]
                            right = dict_max_series(merge_data_cols(future_cols))
                        else:
                            months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                            future_cols = [f"rec_{month}_{metric}_all" for month in months]
                            right = dict_max_series(merge_monthly_all_cols(future_cols))
                        block[f"{source_name}inc"] = safe_divide(left, right)
                    else:
                        future_name = f"rec_{end_month}_{org}_{metric}"
                        diff_series = dt_source[future_name] - dt_source[source_name]
                        block[f"{source_name}inc"] = safe_divide(dt_source[source_name], diff_series)
        return pd.DataFrame(block, index=data.index)

    def build_org_inc(dt_source, desc_suffix=""):
        dm_inc = [1, 3, 6, 12]
        block = {}
        for i in progress_iter(dm_inc, desc=f"生成机构环比{desc_suffix}", total=len(dm_inc), enabled=show_progress):
            for org in org_type4[:-1]:
                for status in ["suc", "fail"]:
                    source_name = f"rec_{i}_{org}_{status}_out_ogn"
                    if org != "all":
                        months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                        future_cols = [f"pre_{month}_{org}_{status}_out_max" for month in months]
                        right_series = dict_len_series(merge_data_cols(future_cols))
                    else:
                        months = list(range(13, 24)) if i == 12 else list(range(i + 1, 2 * i + 1))
                        future_cols = [f"rec_{month}_{status}_out_max_all" for month in months]
                        right_series = dict_len_series(merge_monthly_all_cols(future_cols))
                    block[f"{source_name}inc"] = safe_divide(dt_source[source_name], right_series)
        return pd.DataFrame(block, index=data.index)

    def build_basic_yoy(dt_source, desc_suffix=""):
        dm_yoy = [1, 3, 6]
        block = {}
        for i in progress_iter(dm_yoy, desc=f"生成基础变量同比{desc_suffix}", total=len(dm_yoy), enabled=show_progress):
            compare_months = list(range(13, 13 + i))
            for metric in state_add:
                if metric in ["suc_out_cnt", "suc_out_amt"]:
                    target_orgs = org_type4
                    all_orgs = org_type1
                elif metric.endswith("max"):
                    target_orgs = org_type4[:-1]
                    all_orgs = org_type2
                else:
                    target_orgs = org_type4[:-1]
                    all_orgs = org_type2

                for org in target_orgs:
                    source_name = f"rec_{i}_{org}_{metric}"
                    target_name = f"{source_name}yoy"
                    if metric.endswith("max"):
                        left = dict_max_series(dt_source[source_name])
                        if org == "all":
                            compare_cols = [f"rec_{month}_{metric}_all" for month in compare_months]
                            right = dict_max_series(merge_monthly_all_cols(compare_cols))
                        else:
                            compare_cols = [f"pre_{month}_{org}_{metric}" for month in compare_months]
                            right = dict_max_series(merge_data_cols(compare_cols))
                        block[target_name] = safe_divide(left, right)
                    else:
                        if org == "all":
                            compare_cols = [
                                f"pre_{month}_{sub_org}_{metric}"
                                for month in compare_months
                                for sub_org in all_orgs
                            ]
                        else:
                            compare_cols = [f"pre_{month}_{org}_{metric}" for month in compare_months]
                        right = data[compare_cols].sum(axis=1)
                        block[target_name] = safe_divide(dt_source[source_name], right)
        return pd.DataFrame(block, index=data.index)

    def build_org_yoy(dt_source, desc_suffix=""):
        dm_yoy = [1, 3, 6]
        block = {}
        for i in progress_iter(dm_yoy, desc=f"生成机构同比{desc_suffix}", total=len(dm_yoy), enabled=show_progress):
            compare_months = list(range(13, 13 + i))
            for org in org_type4[:-1]:
                for status in ["suc", "fail"]:
                    source_name = f"rec_{i}_{org}_{status}_out_ogn"
                    target_name = f"{source_name}yoy"
                    if org != "all":
                        compare_cols = [f"pre_{month}_{org}_{status}_out_max" for month in compare_months]
                        right_series = dict_len_series(merge_data_cols(compare_cols))
                    else:
                        compare_cols = [f"rec_{month}_{status}_out_max_all" for month in compare_months]
                        right_series = dict_len_series(merge_monthly_all_cols(compare_cols))
                    block[target_name] = safe_divide(dt_source[source_name], right_series)
        return pd.DataFrame(block, index=data.index)

    def build_bcto(dt_source, cross_months, desc_suffix=""):
        block = {}
        for month in progress_iter(cross_months, desc=f"生成业务交叉指标{desc_suffix}", total=len(cross_months), enabled=show_progress):
            temp_cross = pd.DataFrame(index=data.index)
            for org in org_type2:
                cols = [
                    f"rec_{month}_{org}_suc_out_amt",
                    f"rec_{month}_{org}_fail_out_amt",
                    f"rec_{month}_{org}_suc_in_amt",
                ]
                temp_cross[org] = dt_source[cols].sum(axis=1)
            block[f"r{month}_bcto"] = (temp_cross[org_type2] != 0).sum(axis=1).ge(2).astype(int)
        return pd.DataFrame(block, index=data.index)

    log_stage("构建旧口径 rec_n（供第7/8层和同比使用）...", show_progress)
    legacy_base = build_rec_base(date_legacy, include_current=False, desc_suffix="（旧口径）")
    legacy_org = build_org_features(legacy_base, date_legacy, desc_suffix="（旧口径）")
    legacy_dt = pd.concat([legacy_base, legacy_org], axis=1)

    basic_inc_dt = build_basic_inc(legacy_dt, desc_suffix="（旧口径）")
    org_inc_dt = build_org_inc(legacy_dt, desc_suffix="（旧口径）")
    basic_yoy_dt = build_basic_yoy(legacy_dt, desc_suffix="（旧口径）")
    org_yoy_dt = build_org_yoy(legacy_dt, desc_suffix="（旧口径）")

    log_stage("构建含当月 rec_n（供第1/2/3/4/6/9层使用）...", show_progress)
    current_base = build_rec_base(date_current, include_current=True, desc_suffix="（含当月）")
    current_avg_prop = build_avg_prop(current_base, date_current, desc_suffix="（含当月）")
    current_partial = pd.concat([current_base, current_avg_prop], axis=1)
    current_org = build_org_features(current_partial, date_current, desc_suffix="（含当月）")
    current_dt = pd.concat([current_partial, current_org], axis=1)

    compare_dt = build_time_compare(current_dt, [1, 3, 6, 12, 24], desc_suffix="（含当月）")
    diff_dt = build_diff_metrics(current_dt, date_current, desc_suffix="（含当月）")
    bcto_dt = build_bcto(current_dt, [0, 1, 3, 6, 12, 18, 24], desc_suffix="（含当月）")
    time_dt = build_time_features()

    dt = pd.concat(
        [
            current_dt,
            compare_dt,
            diff_dt,
            time_dt,
            basic_inc_dt,
            org_inc_dt,
            basic_yoy_dt,
            org_yoy_dt,
            bcto_dt,
        ],
        axis=1,
    )

    # 选取所需变量：保持与旧版一致，缺列时直接报错，避免静默输出残缺文件
    cols_all = pd.read_table("变量列表.csv", encoding="utf-8")
    cols_lst = list(cols_all["var_name"][1:])
    dt_all = dt[cols_lst].copy()
    object_cols = [col for col in dt_all.columns if dt_all[col].dtype == "O"]
    for col in progress_iter(object_cols, desc="将 dict 型字段转数值", total=len(object_cols), enabled=show_progress):
        dt_all[col] = [dict_max(value) for value in dt_all[col]]

    for month in [0, 1, 3, 6, 12, 18, 24]:
        for org in org_type3:
            dt_all[f"r{month}_{org}_sf_out_mad"] = dict_max_series(dt[f"rec_{month}_{org}_suc_out_max"]) - dict_max_series(dt[f"rec_{month}_{org}_fail_out_max"])

    dt_all_res = pd.concat(
        [data[["index", "idcard", "dateBack", "cardnum"]].reset_index(drop=True), dt_all.reset_index(drop=True)],
        axis=1,
    )

    col_1000_all = pd.read_csv("1000list.csv")
    col_1000_lst = list(col_1000_all["英文"])
    df_1000 = dt_all_res[["index", "idcard", "dateBack"] + col_1000_lst].copy()
    df_1000["fee_status"] = df_1000["cardnum"].fillna(0).gt(0).astype(int)

    output_file = file_name if str(file_name).lower().endswith(".csv") else f"{file_name}.csv"
    df_1000.to_csv(output_file)
    log_stage(f"结果已保存到 {output_file}", show_progress)

    return df_1000

In [ ]:
df_new = build_df_new(
    df,
    key_cols=("idcard", "dateBack"),
    card_col="card",
    serialize_max=False,
    show_progress=True,
)

result_df = get(
    df_new,
    "文件名",
    show_progress=True,
    pre_aggregated=True,
)

# del df
import gc
gc.collect()
result_df.head()